In [ ]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 3: Codificación de variables temporales (datetime) en seno/coseno,
          transformación de dirección y ángulo a seno/coseno,
          y codificación de variables categóricas (Estacion, Transecto)
          para ML (OneHot) y DL (enteros para embedding).

Entrada: Carpetas con datos imputados:
    - Datos_TFG_outliers/imputed_by_station/
    - Datos_TFG_outliers/imputed_global/
Salida:
    - Datos_TFG_outliers/encoded_ml/by_station/
    - Datos_TFG_outliers/encoded_ml/global/
    - Datos_TFG_outliers/encoded_dl/by_station/
    - Datos_TFG_outliers/encoded_dl/global/
    (dentro de cada carpeta, los archivos CSV con las transformaciones aplicadas)
    Además, en la carpeta encoded_dl se guardan archivos JSON con los mapeos de categorías.
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_BY_STATION = os.path.join(BASE_DIR, "imputed_by_station")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_BASE = os.path.join(BASE_DIR, "encoded")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_BASE, "ml", "by_station")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_BASE, "ml", "global")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_BASE, "dl", "by_station")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_BASE, "dl", "global")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_BY_STATION, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Columnas que se esperan en los archivos (además de las que se crearán)
# No todas pueden estar presentes; se manejarán con if.
ORIGINAL_NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def cyclical_encode(series, period):
    """Convierte una serie numérica (con valores en [0, period)) a seno/coseno."""
    rad = 2 * np.pi * series / period
    sin = np.sin(rad)
    cos = np.cos(rad)
    return sin, cos

def add_datetime_features(df):
    """
    A partir del índice datetime, extrae:
    - hour (0-23)
    - dayofyear (1-366)
    - week (1-52)
    - month (1-12)
    - year (entero)
    y los transforma a seno/coseno (excepto year, que se deja lineal).
    Retorna df con nuevas columnas.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    # Extraer componentes
    hour = idx.hour
    dayofyear = idx.dayofyear   # 1-366
    week = idx.isocalendar().week.astype(int)  # 1-53
    month = idx.month
    year = idx.year

    # Codificación cíclica
    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    # dayofyear: periodo 365 (o 366, pero usamos 365 para simplificar)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    # week: periodo 52
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    # month: periodo 12
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    # año como entero (tendencia)
    df['year'] = year

    return df

def add_directional_features(df):
    """
    Convierte Direc. y Angulo (en grados) a seno/coseno.
    Asume que existen esas columnas. Las reemplaza por las nuevas.
    """
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            rad = np.radians(df[col])
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            # Eliminar columna original
            df.drop(columns=[col], inplace=True)
    return df

def encode_categorical_ml(df, cat_cols):
    """
    Aplica OneHotEncoder a las columnas categóricas.
    Retorna df con las dummies añadidas y elimina las originales.
    Nota: Usa pandas get_dummies por simplicidad; preserva el índice.
    """
    if not cat_cols:
        return df
    # Verificar qué columnas existen
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df
    # One-hot con pandas get_dummies (genera columnas con prefijo)
    dummies = pd.get_dummies(df[existing], prefix=existing, dummy_na=False)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df

def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    """
    Convierte las columnas categóricas a enteros (0..n-1) usando factorize.
    Si save_mapping es True, guarda un dict con el mapeo (categoría -> código) en mapping_file (json).
    Retorna df con las columnas reemplazadas por enteros.
    """
    if not cat_cols:
        return df, {}
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col])
        df[col] = codes
        mapping[col] = {str(k): int(v) for v, k in enumerate(uniques)}  # serializable
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping

def process_file(input_path, output_dir_ml, output_dir_dl, is_global=False):
    """
    Procesa un archivo CSV: aplica codificaciones y guarda versiones ML y DL.
    input_path: ruta del archivo de entrada.
    output_dir_ml: carpeta donde guardar la versión ML (one-hot).
    output_dir_dl: carpeta donde guardar la versión DL (enteros).
    is_global: si es True, se asume que el archivo contiene datos de múltiples estaciones.
    """
    station_name = Path(input_path).stem
    print(f"Procesando: {station_name} (global={is_global})")

    # Cargar datos
    df = pd.read_csv(input_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Añadir características temporales
    df = add_datetime_features(df)

    # Añadir características direccionales (sin/cos)
    df = add_directional_features(df)

    # Hacer una copia para ML y otra para DL
    df_ml = df.copy()
    df_dl = df.copy()

    # Codificación ML: one-hot
    df_ml = encode_categorical_ml(df_ml, CATEGORICAL_COLS)

    # Codificación DL: factorize
    mapping_file = os.path.join(output_dir_dl, f"{station_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df_dl, CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)

    # Guardar archivos
    ml_path = os.path.join(output_dir_ml, f"{station_name}.csv")
    dl_path = os.path.join(output_dir_dl, f"{station_name}.csv")
    df_ml.to_csv(ml_path, index=True)
    df_dl.to_csv(dl_path, index=True)
    print(f"  ML guardado en {ml_path}")
    print(f"  DL guardado en {dl_path}")

def process_folder(input_folder, output_ml, output_dl, is_global=False):
    """Procesa todos los archivos CSV en input_folder."""
    files = list(Path(input_folder).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {input_folder}")
        return
    for f in files:
        process_file(f, output_ml, output_dl, is_global)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("Iniciando codificación de variables...")
    print("="*60)

    # Procesar datos por estación (individual)
    if os.path.exists(INPUT_BY_STATION):
        print("\n--- Datos por estación (individual) ---")
        process_folder(INPUT_BY_STATION, OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, is_global=False)
    else:
        print(f"La carpeta {INPUT_BY_STATION} no existe. Se omite.")

    # Procesar datos globales (todas las estaciones juntas)
    if os.path.exists(INPUT_GLOBAL):
        print("\n--- Datos globales (todas las estaciones) ---")
        process_folder(INPUT_GLOBAL, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)
    else:
        print(f"La carpeta {INPUT_GLOBAL} no existe. Se omite.")

    print("\nProceso completado. Revise las carpetas de salida.")
